# Imports

In [13]:
import os
import sys
import glob
import tempfile
import pandas as pd
from pcapflower import convert_pcap_to_csv

# Inputs

In [14]:
DATA_ROOT = "../../../Datasets"

# Folder names to include (relative to DATA_ROOT, e.g. ["benign", "dos_attack"])
# Leave empty to process all folders found under DATA_ROOT
INCLUDE_FOLDERS = ["CIC_IOT_Dataset_2023"]

# Preprocessing

In [15]:
def find_pcap_folders(root_dir, include=None):
    """Return every folder (recursively) that contains at least one PCAP file.
    If include is non-empty, only searches within those named subfolders of root_dir.
    """
    if include:
        search_roots = [
            os.path.join(root_dir, name)
            for name in include
            if os.path.isdir(os.path.join(root_dir, name))
        ]
    else:
        search_roots = [root_dir]

    found = []
    for search_root in search_roots:
        for dirpath, _, filenames in os.walk(search_root):
            if any(f.endswith((".pcap", ".pcapng")) for f in filenames):
                found.append(dirpath)
    return sorted(found)


def generate_merged_csv_from_pcaps(pcap_dir, output_filename):
    pcap_files = sorted(
        glob.glob(os.path.join(pcap_dir, "*.pcap")) +
        glob.glob(os.path.join(pcap_dir, "*.pcapng"))
    )

    if not pcap_files:
        raise FileNotFoundError(f"No PCAP files found in: {pcap_dir}")

    label = os.path.basename(pcap_dir)
    print(f"[{label}] {len(pcap_files)} PCAP file(s)")

    output_path = os.path.join(pcap_dir, output_filename)
    total_flows = 0
    first_write = True

    with tempfile.TemporaryDirectory(dir=pcap_dir) as tmp_dir:
        for i, pcap_path in enumerate(pcap_files, 1):
            pcap_name = os.path.basename(pcap_path)
            print(f"  [{i}/{len(pcap_files)}] {pcap_name}")

            tmp_csv = os.path.join(tmp_dir, f"{pcap_name}.csv")
            n_flows = convert_pcap_to_csv(pcap_path, tmp_csv, n_jobs= 8)
            print(f"    → {n_flows} flows")

            if n_flows > 0:
                for chunk in pd.read_csv(tmp_csv, chunksize=50_000):
                    chunk["label"] = label
                    chunk.to_csv(output_path, mode="w" if first_write else "a",
                                 index=False, header=first_write)
                    total_flows += len(chunk)
                    first_write = False

    if total_flows == 0:
        raise RuntimeError(f"No flows extracted from: {pcap_dir}")

    print(f"  ✓ {output_path}  ({total_flows} flows)")
    return output_path

In [16]:
pcap_folders = find_pcap_folders(DATA_ROOT, include=INCLUDE_FOLDERS)
print(f"Found {len(pcap_folders)} folder(s) with PCAPs\n")

for folder_path in pcap_folders:
    folder_name = os.path.basename(folder_path)
    generate_merged_csv_from_pcaps(folder_path, f"merged_{folder_name}.csv")

Found 7 folder(s) with PCAPs

[benign] 4 PCAP file(s)
  [1/4] BenignTraffic.pcap
    → 226098 flows
  [2/4] BenignTraffic1.pcap
    → 74471 flows
  [3/4] BenignTraffic2.pcap
    → 80540 flows
  [4/4] BenignTraffic3.pcap
    → 34479 flows
  ✓ ../../../Datasets/CIC_IOT_Dataset_2023/benign/merged_benign.csv  (415588 flows)
[bruteforce] 1 PCAP file(s)
  [1/1] DictionaryBruteForce.pcap
    → 9730 flows
  ✓ ../../../Datasets/CIC_IOT_Dataset_2023/bruteforce/merged_bruteforce.csv  (9730 flows)
[malware] 77 PCAP file(s)
  [1/77] Backdoor_Malware.pcap
    → 2638 flows
  [2/77] Mirai-greeth_flood.pcap
    → 5970 flows
  [3/77] Mirai-greeth_flood1.pcap
    → 11968 flows
  [4/77] Mirai-greeth_flood10.pcap
    → 9119 flows
  [5/77] Mirai-greeth_flood11.pcap
    → 9566 flows
  [6/77] Mirai-greeth_flood12.pcap
    → 7154 flows
  [7/77] Mirai-greeth_flood13.pcap
    → 12094 flows
  [8/77] Mirai-greeth_flood14.pcap
    → 3743 flows
  [9/77] Mirai-greeth_flood15.pcap
    → 3323 flows
  [10/77] Mirai-gree